# Módulo 10 · Aula 05 — Arquitetura de ETL e Qualidade

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O pipeline rodou às 3h e falhou na linha 40.212, por causa de um CEP com letra. Perdemos a carga inteira da noite. E hoje de manhã eu descobri que ele vinha gravando `quantidade` negativa há duas semanas — ninguém percebeu porque não deu erro."*

Duas falhas opostas, e as duas erradas:

| O que aconteceu | Deveria ter acontecido |
|-----------------|------------------------|
| Uma linha ruim derrubou 40 mil boas | A linha ruim vai para a quarentena |
| Duas semanas de dado absurdo passaram | O pipeline barra o que não faz sentido |

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | ETL vs ELT | E qual faz sentido hoje |
| 2 | **As três camadas na prática** | 🎯 Código, não diagrama |
| 3 | 🔴 **Idempotência** | Rodar de novo é seguro |
| 4 | 🔴 **Quarentena** | Dado ruim não some nem derruba |
| 5 | Contratos com Pydantic | O M04 e o M06 se encontram |
| 6 | Verificações de qualidade | As seis que valem sempre |
| 7 | Reprocessamento | Consertar o passado |
| 8 | Linhagem e metadados | De onde veio cada número |

> 🎯 **Esta é a aula que junta o módulo.** Você vai construir um pipeline completo que roda, falha, se recupera e é reprocessado — tudo executado de verdade.

## ⚙️ Preparação

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 10
# ═══════════════════════════════════════════════════════════════
import json
import os
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import time
import warnings
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("pandas", "pandas"), ("numpy", "numpy"),
               ("pyarrow", "pyarrow"), ("polars", "polars"),
               ("duckdb", "duckdb")]:
    _garantir(_p, _m)

import numpy as np
import pandas as pd

TEM_POLARS = _garantir("polars", "polars")
TEM_DUCKDB = _garantir("duckdb", "duckdb")
TEM_ARROW = _garantir("pyarrow", "pyarrow")

pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print(f"pandas {pd.__version__} · numpy {np.__version__}")
if TEM_POLARS:
    import polars as pl
    print(f"polars {pl.__version__}")
if TEM_DUCKDB:
    import duckdb
    print(f"duckdb {duckdb.__version__}")


# ═══════════════════════════════════════════════════════════════
#  Dados da Aurora — gerados de forma REPRODUTÍVEL
# ═══════════════════════════════════════════════════════════════
SEMENTE = 20260813
rng = np.random.default_rng(SEMENTE)
random.seed(SEMENTE)

CIDADES = ["Campinas", "São Paulo", "Valinhos", "Sumaré",
           "Indaiatuba", "Jundiaí", "Hortolândia"]
CANAIS = ["site", "app", "marketplace"]
CATEGORIAS = {
    "NB": ("Notebooks", 1800, 4200),
    "MO": ("Monitores", 700, 2200),
    "PE": ("Periféricos", 40, 400),
    "AR": ("Armazenamento", 180, 900),
}


def gerar_produtos(n: int = 60) -> pd.DataFrame:
    linhas = []
    for i in range(n):
        prefixo = list(CATEGORIAS)[i % len(CATEGORIAS)]
        categoria, minimo, maximo = CATEGORIAS[prefixo]
        preco = round(float(rng.uniform(minimo, maximo)), 2)
        linhas.append({
            "sku": f"{prefixo}-{1000 + i}",
            "nome": f"{categoria[:-1]} modelo {i:03d}",
            "categoria": categoria,
            "preco": preco,
            "custo": round(preco * float(rng.uniform(0.55, 0.85)), 2),
            "estoque": int(rng.integers(0, 200)),
        })
    return pd.DataFrame(linhas)


def gerar_vendas(n: int = 50_000, dias: int = 180,
                 produtos: pd.DataFrame | None = None) -> pd.DataFrame:
    """Vendas sintéticas com sazonalidade e um pouco de sujeira."""
    produtos = gerar_produtos() if produtos is None else produtos
    fim = datetime(2026, 8, 1, tzinfo=timezone.utc)
    inicio = fim - timedelta(days=dias)

    idx = rng.integers(0, len(produtos), n)
    escolhidos = produtos.iloc[idx].reset_index(drop=True)

    # 📈 Sazonalidade: mais vendas no fim de semana e no fim do mês
    deslocamento = rng.integers(0, dias, n)
    datas = pd.to_datetime(inicio) + pd.to_timedelta(deslocamento, unit="D")
    datas = datas + pd.to_timedelta(rng.integers(0, 86400, n), unit="s")

    return pd.DataFrame({
        "pedido_id": 100_000 + np.arange(n),
        "data": datas,
        "sku": escolhidos["sku"],
        "categoria": escolhidos["categoria"],
        "cidade": rng.choice(CIDADES, n, p=[.28, .22, .12, .12, .11, .09, .06]),
        "canal": rng.choice(CANAIS, n, p=[.55, .30, .15]),
        "quantidade": rng.integers(1, 6, n),
        "preco_unitario": escolhidos["preco"],
        "custo_unitario": escolhidos["custo"],
        "status": rng.choice(["pago", "pendente", "cancelado"], n, p=[.82, .10, .08]),
        "frete": np.round(rng.uniform(0, 45, n), 2),
    })


# ═══════════════════════════════════════════════════════════════
#  Medição
# ═══════════════════════════════════════════════════════════════

def cronometrar(funcao, repeticoes: int = 1):
    """Devolve (resultado, milissegundos_medios)."""
    inicio = time.perf_counter()
    resultado = None
    for _ in range(repeticoes):
        resultado = funcao()
    return resultado, (time.perf_counter() - inicio) * 1000 / repeticoes


def comparar(casos: list[tuple[str, callable]], repeticoes: int = 1,
             rotulo: str = "abordagem"):
    """Mede várias abordagens e mostra o ganho relativo."""
    medidos = []
    for nome, funcao in casos:
        _, ms = cronometrar(funcao, repeticoes)
        medidos.append((nome, ms))
    melhor = min(m for _, m in medidos)
    largura = max(len(n) for n, _ in medidos) + 2
    print(f"{rotulo:<{largura}}{'tempo':>12}   {'vs melhor':>10}")
    print("─" * (largura + 26))
    for nome, ms in medidos:
        barra = "█" * max(1, int(ms / melhor))
        print(f"{nome:<{largura}}{ms:>9.1f} ms   {ms / melhor:>8.1f}×  {barra[:26]}")
    return medidos


def tamanho(n: int) -> str:
    for unidade in ("B", "KB", "MB", "GB"):
        if n < 1024 or unidade == "GB":
            return f"{n:,.1f} {unidade}" if unidade != "B" else f"{n:,} B"
        n /= 1024
    return ""


def memoria(df: pd.DataFrame) -> int:
    return int(df.memory_usage(deep=True).sum())


def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]) -> None:
    """Tabela ASCII alinhada (marcadores ASCII, não emoji — M03)."""
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


print("\n✅ `gerar_vendas()`, `comparar()`, `memoria()`, `tabela()` prontos")
print(f"   semente fixa ({SEMENTE}) — os números são reprodutíveis")

## 1. ETL vs ELT

In [ ]:
BASE = preparar("aula_10_05")

print("""
   ETL — Extract, Transform, Load
   ══════════════════════════════
   extrai → TRANSFORMA fora do destino → carrega o resultado
   · o destino recebe só o dado pronto
   · 🔴 se a transformação tiver bug, o dado cru já não existe

   ELT — Extract, Load, Transform
   ══════════════════════════════
   extrai → carrega o CRU → transforma DENTRO do destino
   · o cru fica guardado
   · exige um destino que aguente transformar (warehouse, DuckDB)

🎯 O ELT venceu porque o armazenamento ficou barato.

   Guardar o cru custa centavos; descobrir que a transformação de seis
   meses atrás estava errada e não poder recalcular custa a confiança
   no número.

   💭 As camadas bronze/prata/ouro (aula 10_01) SÃO um ELT: você carrega
      o cru primeiro (bronze) e transforma depois, em etapas.
""")

## 2. 🎯 As três camadas na prática

In [ ]:
LAGO = BASE / "lago"
for camada in ("bronze", "prata", "ouro", "quarentena", "estado"):
    (LAGO / camada).mkdir(parents=True)

print("Estrutura do lago:\n")
for camada in ("bronze", "prata", "ouro", "quarentena", "estado"):
    print(f"   lago/{camada}/")

print("""
   bronze/data_ingestao=AAAA-MM-DD/   como chegou, IMUTÁVEL
   prata/                             limpo, tipado, deduplicado
   ouro/                              agregado para consumo
   quarentena/data=AAAA-MM-DD/        o que não passou, COM O MOTIVO
   estado/                            marca d'água e metadados
""")

In [ ]:
# ═══ A fonte: dados com problemas REAIS ═══
def gerar_lote_sujo(n: int = 5_000, semente: int = 1) -> pd.DataFrame:
    """Um lote com os defeitos que aparecem de verdade."""
    local = np.random.default_rng(semente)
    df = gerar_vendas(n=n, dias=30)

    # 🔴 Os defeitos, um a um
    df.loc[local.choice(df.index, 60, replace=False), "quantidade"] = -1
    df.loc[local.choice(df.index, 40, replace=False), "preco_unitario"] = 0
    df.loc[local.choice(df.index, 30, replace=False), "cidade"] = None
    df.loc[local.choice(df.index, 25, replace=False), "status"] = "PAGO"     # caixa
    df.loc[local.choice(df.index, 20, replace=False), "sku"] = "  nb-1000 "  # sujo
    df.loc[local.choice(df.index, 15, replace=False), "preco_unitario"] = 999_999.0

    # duplicatas: o mesmo pedido reenviado
    duplicadas = df.sample(50, random_state=semente)
    return pd.concat([df, duplicadas], ignore_index=True)


lote = gerar_lote_sujo()
print(f"lote: {len(lote):,} linhas\n")
print("Defeitos plantados:")
print(f"   quantidade <= 0      : {(lote['quantidade'] <= 0).sum():>4}")
print(f"   preço <= 0           : {(lote['preco_unitario'] <= 0).sum():>4}")
print(f"   cidade nula          : {lote['cidade'].isna().sum():>4}")
print(f"   status fora do padrão: {(~lote['status'].isin(['pago','pendente','cancelado'])).sum():>4}")
print(f"   sku com espaço/caixa : {(lote['sku'] != lote['sku'].str.strip().str.upper()).sum():>4}")
print(f"   pedido_id duplicado  : {lote['pedido_id'].duplicated().sum():>4}")

## 3. 🔴 Bronze — imutável e idempotente

In [ ]:
import hashlib


def gravar_bronze(df: pd.DataFrame, origem: str, data_ingestao: date) -> dict:
    """Grava o lote CRU, sem tocar em nada.

    🔑 TRÊS PROPRIEDADES:

    1. IMUTÁVEL — nunca reescrevemos um arquivo já gravado.
    2. PARTICIONADO por data de ingestão — permite reprocessar um dia.
    3. IDEMPOTENTE — rodar de novo no mesmo dia SUBSTITUI a partição,
       não acumula. É o que torna seguro repetir a carga.

    ⚠️ E gravamos o HASH do conteúdo. Ele responde depois a pergunta
       "este arquivo é o mesmo que eu processei?" — sem precisar
       comparar linha a linha.
    """
    pasta = LAGO / "bronze" / f"origem={origem}" / f"data_ingestao={data_ingestao}"
    if pasta.exists():
        shutil.rmtree(pasta)                 # 🔑 substitui, não acumula
    pasta.mkdir(parents=True)

    arquivo = pasta / "dados.parquet"
    df.to_parquet(arquivo, index=False)

    digest = hashlib.sha256(arquivo.read_bytes()).hexdigest()[:16]
    manifesto = {
        "origem": origem,
        "data_ingestao": str(data_ingestao),
        "linhas": len(df),
        "colunas": list(df.columns),
        "sha256": digest,
        "gravado_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    (pasta / "_manifesto.json").write_text(
        json.dumps(manifesto, indent=2, ensure_ascii=False), encoding="utf-8")
    return manifesto


hoje = date(2026, 8, 13)
manifesto = gravar_bronze(lote, "erp", hoje)
print(json.dumps(manifesto, indent=2, ensure_ascii=False)[:400])

# Idempotência do bronze
m2 = gravar_bronze(lote, "erp", hoje)
arquivos = list((LAGO / "bronze").rglob("*.parquet"))
print(f"\nrodou 2× → {len(arquivos)} arquivo(s)  ✅ substituiu, não acumulou")
print(f"hash igual: {manifesto['sha256'] == m2['sha256']}")

> 🔑 **O manifesto ao lado do arquivo é o que torna o lago navegável.**
>
> Sem ele, `bronze/` é uma pasta de Parquets misteriosos. Com ele, você responde sem abrir nada: *de onde veio, quantas linhas, quando chegou, e é o mesmo conteúdo de antes?*
>
> 💭 É o mesmo princípio dos metadados do relatório na aula 10_02: **todo dado publicado vem com a receita de como chegou ali.**

## 4. Contratos com Pydantic

In [ ]:
from pydantic import BaseModel, Field, ValidationError, field_validator

STATUS_VALIDOS = {"pago", "pendente", "cancelado"}
PRECO_MAXIMO = 100_000.0


class VendaValida(BaseModel):
    """O CONTRATO da camada prata.

    💭 É o mesmo Pydantic do M04 (validação interna) e do M06 (fronteira
       da API). Aqui ele é a fronteira do PIPELINE — e o papel é igual:
       o que não passa não entra.

    🔑 A diferença é o que fazer com quem falha. Na API, você devolve
       422 e o cliente corrige. Aqui não há cliente para corrigir —
       por isso existe a quarentena.
    """
    model_config = {"str_strip_whitespace": True}

    pedido_id: int = Field(gt=0)
    data: datetime
    sku: str = Field(min_length=5, max_length=20)
    categoria: str
    cidade: str | None = None
    canal: str
    quantidade: int = Field(gt=0, le=1000)
    preco_unitario: float = Field(gt=0, le=PRECO_MAXIMO)
    custo_unitario: float = Field(ge=0)
    status: str
    frete: float = Field(ge=0)

    @field_validator("sku", mode="before")
    @classmethod
    def normalizar_sku(cls, v):
        # 🔑 mode="before": normaliza ANTES das restrições (M06)
        return v.strip().upper() if isinstance(v, str) else v

    @field_validator("status", mode="before")
    @classmethod
    def normalizar_status(cls, v):
        return v.strip().lower() if isinstance(v, str) else v

    @field_validator("status")
    @classmethod
    def status_conhecido(cls, v):
        if v not in STATUS_VALIDOS:
            raise ValueError(f"status '{v}' fora de {sorted(STATUS_VALIDOS)}")
        return v

    @field_validator("cidade", mode="before")
    @classmethod
    def cidade_padrao(cls, v):
        # DECISÃO: cidade nula vira grupo próprio (aula 10_02)
        return v if v not in (None, "", "nan") else "não informado"


print("✅ contrato definido\n")
exemplo = VendaValida(
    pedido_id=1, data="2026-01-05T10:00:00Z", sku="  nb-1000  ",
    categoria="Notebooks", cidade=None, canal="site", quantidade=2,
    preco_unitario=2599.9, custo_unitario=2120.0, status="PAGO", frete=25.0)
print(f"   sku normalizado    : {exemplo.sku!r}")
print(f"   status normalizado : {exemplo.status!r}")
print(f"   cidade nula        : {exemplo.cidade!r}")

## 5. 🔴 Quarentena — dado ruim não some nem derruba

In [ ]:
def validar_lote(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Separa o que passa do que não passa.

    🔴 A REGRA: uma linha ruim NUNCA derruba o lote, e NUNCA some.

       · derrubar o lote → 40 mil linhas boas perdidas por causa de uma
       · descartar em silêncio → o número fica errado e ninguém sabe

       A quarentena é a terceira via: a linha sai do fluxo principal
       COM O MOTIVO registrado, e alguém pode consertá-la depois.
    """
    validas, rejeitadas = [], []

    for registro in df.to_dict("records"):
        try:
            validas.append(VendaValida(**registro).model_dump())
        except ValidationError as erro:
            motivos = "; ".join(
                f"{'.'.join(str(p) for p in e['loc'])}: {e['msg']}"
                for e in erro.errors())
            rejeitadas.append({**registro,
                               "_motivo": motivos,
                               "_rejeitado_em": datetime.now(timezone.utc)
                                                .isoformat(timespec="seconds")})

    return pd.DataFrame(validas), pd.DataFrame(rejeitadas)


bronze = pd.read_parquet(
    LAGO / "bronze" / "origem=erp" / f"data_ingestao={hoje}" / "dados.parquet")

validas, rejeitadas = validar_lote(bronze)

print(f"   entraram  : {len(bronze):>6,}")
print(f"   válidas   : {len(validas):>6,}  ({len(validas)/len(bronze):.1%})")
print(f"   rejeitadas: {len(rejeitadas):>6,}  ({len(rejeitadas)/len(bronze):.1%})")

print("\nMotivos da rejeição:\n")
motivos = (rejeitadas["_motivo"]
           .str.split(";").str[0].str.strip()
           .value_counts())
for motivo, n in motivos.items():
    print(f"   {n:>4}× {motivo}")

In [ ]:
# A quarentena é gravada, com contexto
def gravar_quarentena(df: pd.DataFrame, data_ingestao: date) -> Path | None:
    if df.empty:
        return None
    pasta = LAGO / "quarentena" / f"data={data_ingestao}"
    pasta.mkdir(parents=True, exist_ok=True)
    arquivo = pasta / "rejeitadas.parquet"
    df.astype({c: str for c in df.columns if df[c].dtype == object}).to_parquet(
        arquivo, index=False)
    return arquivo


arquivo_q = gravar_quarentena(rejeitadas, hoje)
print(f"✅ {arquivo_q.relative_to(BASE)}  ({tamanho(arquivo_q.stat().st_size)})\n")
print(rejeitadas[["pedido_id", "sku", "quantidade", "preco_unitario",
                  "status", "_motivo"]].head(4).to_string(index=False))

print("""
🎯 AGORA ALGUÉM PODE AGIR.

   O time de operações abre a quarentena, vê que 60 pedidos vieram
   com quantidade -1, descobre o bug no sistema de origem, corrige, e
   REPROCESSA aquele dia.

   💭 Sem quarentena, essas 60 linhas teriam duas opções ruins:
      derrubar a carga, ou desaparecer sem deixar rastro.
""")

## 6. Prata — limpo, tipado, deduplicado

In [ ]:
def construir_prata(validas: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Deduplica, tipa e enriquece.

    🔑 A ORDEM IMPORTA:
       1. ordenar pela recência
       2. deduplicar por chave natural, mantendo a última
       3. só então calcular as derivadas
    """
    df = validas.copy()
    antes = len(df)

    # 🔑 Deduplicação por CHAVE NATURAL, mantendo a mais recente.
    #    Sem o sort, "last" seria o que o acaso decidir (aula 10_02).
    df = df.sort_values("data").drop_duplicates(subset=["pedido_id"], keep="last")
    duplicadas = antes - len(df)

    # Derivadas do negócio
    df["receita"] = (df["quantidade"] * df["preco_unitario"]).round(2)
    df["margem"] = (df["quantidade"] *
                    (df["preco_unitario"] - df["custo_unitario"])).round(2)
    df["margem_pct"] = (df["margem"] / df["receita"] * 100).round(2)

    # Tipos econômicos (aula 10_03) — 🔴 dinheiro fica em float64
    for coluna in ["categoria", "cidade", "canal", "status"]:
        df[coluna] = df[coluna].astype("category")
    df["quantidade"] = df["quantidade"].astype("int16")

    # Linhagem
    df["_camada"] = "prata"
    df["_processado_em"] = datetime.now(timezone.utc).isoformat(timespec="seconds")

    return df.reset_index(drop=True), {
        "entraram": antes,
        "duplicadas_removidas": duplicadas,
        "saíram": len(df),
    }


prata, info = construir_prata(validas)
print(json.dumps(info, indent=2))
print(f"\nmemória: {tamanho(memoria(prata))}\n")
print(prata[["pedido_id", "sku", "quantidade", "receita", "margem_pct"]]
      .head(4).to_string(index=False))

(LAGO / "prata" / "vendas.parquet").parent.mkdir(exist_ok=True)
prata.to_parquet(LAGO / "prata" / "vendas.parquet", index=False)

## 7. 🎯 Verificações de qualidade

In [ ]:
def verificar_qualidade(df: pd.DataFrame) -> list[dict]:
    """As seis verificações que valem para quase todo conjunto.

    💭 A diferença entre VALIDAÇÃO e VERIFICAÇÃO DE QUALIDADE:

       validação    olha a LINHA — este registro faz sentido?
       qualidade    olha o CONJUNTO — este lote faz sentido?

       Um lote em que toda linha é válida ainda pode estar errado: se
       hoje chegaram 12 vendas e a média é 5.000, algo aconteceu na
       origem — e nenhuma validação de linha pegaria isso.
    """
    resultados = []

    def checar(nome, passou, detalhe="", gravidade="erro"):
        resultados.append({"verificacao": nome, "passou": bool(passou),
                           "detalhe": detalhe, "gravidade": gravidade})

    # 1. COMPLETUDE — o lote não pode estar vazio
    checar("lote não vazio", len(df) > 0, f"{len(df):,} linhas")

    # 2. UNICIDADE — a chave natural é única
    dup = df["pedido_id"].duplicated().sum()
    checar("pedido_id único", dup == 0, f"{dup} duplicados")

    # 3. NÃO-NULOS — as colunas obrigatórias
    obrigatorias = ["pedido_id", "data", "sku", "quantidade", "preco_unitario"]
    nulos = {c: int(df[c].isna().sum()) for c in obrigatorias if df[c].isna().any()}
    checar("colunas obrigatórias preenchidas", not nulos, str(nulos))

    # 4. FAIXA — os valores fazem sentido
    checar("quantidade positiva", (df["quantidade"] > 0).all(),
           f"{(df['quantidade'] <= 0).sum()} fora")
    checar("receita positiva", (df["receita"] > 0).all(),
           f"{(df['receita'] <= 0).sum()} fora")

    # 5. COERÊNCIA — as relações entre colunas
    margem_impossivel = (df["margem"] > df["receita"]).sum()
    checar("margem <= receita", margem_impossivel == 0,
           f"{margem_impossivel} incoerentes")

    # 6. 🔑 VOLUME — comparação com o histórico
    #    Esta é a que pega o problema SILENCIOSO.
    #
    #    🔑 DOIS LIMIARES, e a distinção importa:
    #       · desvio > 50%  → AVISO. Pode ser feriado, Black Friday,
    #         promoção. Merece olhar humano, não parada automática.
    #       · desvio > 90%  → ERRO. Isso não é sazonalidade: é
    #         extração truncada, filtro errado ou origem fora do ar.
    esperado = ler_volume_esperado()
    if esperado:
        variacao = (len(df) - esperado) / esperado
        detalhe = f"{len(df):,} vs {esperado:,} esperadas ({variacao:+.1%})"
        if abs(variacao) > 0.9:
            checar("volume plausível", False, detalhe, gravidade="erro")
        else:
            checar("volume dentro do esperado", abs(variacao) < 0.5,
                   detalhe, gravidade="aviso")

    return resultados


ESTADO = LAGO / "estado" / "historico.json"


def ler_volume_esperado() -> int | None:
    if not ESTADO.exists():
        return None
    historico = json.loads(ESTADO.read_text(encoding="utf-8")).get("volumes", [])
    return int(np.median(historico)) if historico else None


def registrar_volume(n: int) -> None:
    atual = json.loads(ESTADO.read_text(encoding="utf-8")) if ESTADO.exists() else {}
    volumes = atual.get("volumes", [])
    volumes.append(n)
    atual["volumes"] = volumes[-30:]        # últimos 30 dias
    ESTADO.write_text(json.dumps(atual, indent=2), encoding="utf-8")


def mostrar_qualidade(resultados: list[dict]) -> bool:
    falhas = [r for r in resultados if not r["passou"] and r["gravidade"] == "erro"]
    for r in resultados:
        marca = "✅" if r["passou"] else ("🔴" if r["gravidade"] == "erro" else "⚠️ ")
        print(f"   {marca} {r['verificacao']:<38}{r['detalhe']}")
    return not falhas


print("Verificações da camada prata:\n")
resultados = verificar_qualidade(prata)
tudo_bem = mostrar_qualidade(resultados)
registrar_volume(len(prata))
print(f"\n   {'✅ lote aprovado' if tudo_bem else '🔴 lote REPROVADO'}")

> 🔑 **A sexta verificação é a que pega o problema que ninguém percebe.**
>
> As cinco primeiras olham o dado. A de volume olha o **contexto**: hoje chegou 10× menos que o normal? A extração falhou pela metade. Chegou 10× mais? Alguém reprocessou e duplicou.
>
> 💭 **É o mesmo raciocínio da métrica de negócio no M09.** Latência, erro e CPU podem estar perfeitos enquanto as vendas caem a zero. Aqui, toda linha pode ser válida enquanto o lote inteiro está errado.
>
> ⚠️ **E repare na gravidade `aviso`:** um volume atípico nem sempre é erro (feriado, Black Friday). Ele merece atenção humana, não uma parada automática. **Distinguir "pare tudo" de "olhe isto" é o que evita a fadiga de alerta.**

In [ ]:
# Provando: um lote com problema de volume
lote_pequeno = gerar_lote_sujo(n=200, semente=2)
validas_p, _ = validar_lote(lote_pequeno)
prata_p, _ = construir_prata(validas_p)

print("Um lote com 25× menos linhas que o normal:\n")
mostrar_qualidade(verificar_qualidade(prata_p))
print("\n   💭 Toda linha é válida. O LOTE é que está errado.")

## 8. Ouro — agregado para consumo

In [ ]:
def construir_ouro(prata: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """As tabelas que o negócio consome.

    🔑 Cada uma responde a UMA pergunta. Elas são pequenas, rápidas de
       ler, e não exigem que ninguém entenda o modelo transacional.
    """
    d = prata[prata["status"] == "pago"].copy()
    d["mes"] = pd.to_datetime(d["data"]).dt.to_period("M").astype(str)
    d["dia"] = pd.to_datetime(d["data"]).dt.date

    return {
        "faturamento_mensal": (
            d.groupby(["mes", "categoria"], observed=True)
            .agg(pedidos=("pedido_id", "nunique"),
                 itens=("quantidade", "sum"),
                 receita=("receita", "sum"),
                 margem=("margem", "sum"))
            .reset_index()),

        "vendas_diarias": (
            d.groupby("dia", observed=True)
            .agg(pedidos=("pedido_id", "nunique"),
                 receita=("receita", "sum"))
            .reset_index()),

        "ranking_produtos": (
            d.groupby("sku", observed=True)
            .agg(itens=("quantidade", "sum"), receita=("receita", "sum"))
            .sort_values("receita", ascending=False)
            .head(20).reset_index()),
    }


ouro = construir_ouro(prata)
for nome, df in ouro.items():
    df.to_parquet(LAGO / "ouro" / f"{nome}.parquet", index=False)
    print(f"   {nome:<24}{len(df):>5} linhas   "
          f"{tamanho((LAGO / 'ouro' / f'{nome}.parquet').stat().st_size):>10}")

print("\nfaturamento_mensal:\n")
print(ouro["faturamento_mensal"].to_string(index=False))

In [ ]:
# 🔑 CONCILIAÇÃO: o ouro tem que bater com a prata
receita_prata = prata[prata["status"] == "pago"]["receita"].sum()
receita_ouro = ouro["faturamento_mensal"]["receita"].sum()

print("Conciliação entre camadas:\n")
print(f"   prata (status=pago): R$ {receita_prata:>14,.2f}")
print(f"   ouro  (soma)       : R$ {receita_ouro:>14,.2f}")
print(f"   diferença          : R$ {abs(receita_prata - receita_ouro):>14,.2f}")
assert abs(receita_prata - receita_ouro) < 0.01, "🔴 o ouro não bate com a prata"
print("\n   ✅ bateu — e o `assert` garante que continuará batendo")

print("""
🔑 ESTE ASSERT VALE MAIS QUE PARECE.

   No dia em que alguém adicionar um `groupby` que descarta nulos
   (aula 10_02), o pipeline FALHA em vez de publicar um número menor
   sem avisar.

   💭 É a diferença entre um erro que aparece e um erro que se
      esconde no relatório da diretoria.
""")

## 9. 🎯 O pipeline completo

In [ ]:
class Pipeline:
    """O ETL do Atlas, de ponta a ponta.

    🎯 Cada etapa é IDEMPOTENTE: rodar duas vezes com a mesma entrada
       produz o mesmo estado final.
    """

    def __init__(self, lago: Path):
        self.lago = lago
        self.registro: list[dict] = []

    def _anotar(self, etapa: str, **campos):
        entrada = {"etapa": etapa,
                   "em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                   **campos}
        self.registro.append(entrada)
        detalhes = " · ".join(f"{k}={v}" for k, v in campos.items())
        print(f"   [{etapa:<12}] {detalhes}")

    def executar(self, df_origem: pd.DataFrame, origem: str,
                 data_ingestao: date) -> dict:
        print(f"▶ pipeline · origem={origem} · data={data_ingestao}\n")

        # 1. BRONZE
        manifesto = gravar_bronze(df_origem, origem, data_ingestao)
        self._anotar("bronze", linhas=manifesto["linhas"],
                     sha=manifesto["sha256"][:8])

        # 2. VALIDAÇÃO → prata + quarentena
        bruto = pd.read_parquet(
            self.lago / "bronze" / f"origem={origem}" /
            f"data_ingestao={data_ingestao}" / "dados.parquet")
        validas, rejeitadas = validar_lote(bruto)
        if not rejeitadas.empty:
            gravar_quarentena(rejeitadas, data_ingestao)
        self._anotar("validacao", validas=len(validas), quarentena=len(rejeitadas))

        # 3. PRATA
        prata, info = construir_prata(validas)
        prata.to_parquet(self.lago / "prata" / "vendas.parquet", index=False)
        self._anotar("prata", linhas=len(prata),
                     duplicadas=info["duplicadas_removidas"])

        # 4. QUALIDADE — 🔴 o portão
        resultados = verificar_qualidade(prata)
        falhas = [r for r in resultados
                  if not r["passou"] and r["gravidade"] == "erro"]
        self._anotar("qualidade", verificacoes=len(resultados), falhas=len(falhas))
        if falhas:
            # 🔴 NÃO publica o ouro se a prata reprovou.
            #    Melhor o painel com o dado de ontem do que com o errado de hoje.
            self._anotar("ABORTADO", motivo=falhas[0]["verificacao"])
            return {"sucesso": False, "falhas": falhas, "registro": self.registro}

        # 5. OURO
        tabelas = construir_ouro(prata)
        for nome, tabela in tabelas.items():
            tabela.to_parquet(self.lago / "ouro" / f"{nome}.parquet", index=False)
        self._anotar("ouro", tabelas=len(tabelas))

        registrar_volume(len(prata))
        return {"sucesso": True, "registro": self.registro}


pipeline = Pipeline(LAGO)
resultado = pipeline.executar(gerar_lote_sujo(n=5_000, semente=10),
                              "erp", date(2026, 8, 14))
print(f"\n   {'✅ sucesso' if resultado['sucesso'] else '🔴 abortado'}")

In [ ]:
# ⚠️ Primeiro: um problema que o pipeline CONSERTA sozinho
print("Cenário 1 — bug na origem: 1.500 pedidos com o mesmo id\n")

lote_ids_repetidos = gerar_lote_sujo(n=3_000, semente=11)
lote_ids_repetidos.loc[lote_ids_repetidos.index[:1500], "pedido_id"] = 999999

pipeline2 = Pipeline(LAGO)
r = pipeline2.executar(lote_ids_repetidos, "erp", date(2026, 8, 15))
print(f"\n   {'✅ sucesso' if r['sucesso'] else '🔴 ABORTADO'}")

> ⚠️ **Passou — e por um motivo que vale entender.**
>
> A verificação `pedido_id único` não acusou nada porque a **deduplicação da prata já havia resolvido**: os 1.500 registros com o mesmo id viraram 1 antes de a qualidade olhar.
>
> 🔑 **A ordem das etapas define o que cada verificação consegue ver.** Uma verificação depois da limpeza mede o resultado da limpeza, não a entrada.
>
> 💭 **Isso é bom ou ruim?** Depende da pergunta:
>
> | Você quer saber… | Onde verificar |
> |------------------|----------------|
> | "o dado publicado está consistente?" | **depois** da limpeza (é o que temos) |
> | "a origem está mandando lixo?" | **antes**, no bronze |
>
> **As duas perguntas importam.** Perder 1.499 registros silenciosamente é exatamente o tipo de coisa que se descobre três meses depois — e a resposta é olhar o `duplicadas=` do registro da etapa prata, que o pipeline já anota.

In [ ]:
# 🔴 Agora um problema que o pipeline NÃO pode consertar
print("Cenário 2 — a extração falhou na metade: chegaram 120 linhas\n")

lote_truncado = gerar_lote_sujo(n=120, semente=12)

pipeline3 = Pipeline(LAGO)
r = pipeline3.executar(lote_truncado, "erp", date(2026, 8, 16))
print(f"\n   {'✅ sucesso' if r['sucesso'] else '🔴 ABORTADO'}")
if not r["sucesso"]:
    for f in r["falhas"]:
        print(f"      🔴 {f['verificacao']}: {f['detalhe']}")

print("""
🎯 O OURO NÃO FOI REESCRITO.

   O painel continua mostrando o dado de ontem — correto — em vez do
   dado de hoje, incompleto.

   💭 Essa é a decisão certa quase sempre: DADO VELHO E CERTO É MELHOR
      QUE DADO NOVO E ERRADO. E o alerta (M09) avisa que houve
      problema, para alguém investigar de manhã.

   ⚠️ Repare que nenhuma LINHA estava errada. As 120 são perfeitamente
      válidas — o que está errado é o CONJUNTO. Só a verificação de
      volume pega isso.
""")

## 10. Reprocessamento

In [ ]:
print("""
   O bug na origem foi corrigido. Como consertar o histórico?

   🔑 É AQUI QUE A CAMADA BRONZE SE PAGA.

   Como o bronze é imutável e particionado por data, reprocessar é:

      1. ler bronze/data_ingestao=2026-08-15/
      2. rodar a versão CORRIGIDA da transformação
      3. sobrescrever prata e ouro daquele período

   Sem o bronze, você teria que pedir os dados ao fornecedor de novo —
   se ele ainda os tiver.
""")


def reprocessar(lago: Path, origem: str, de: date, ate: date) -> dict:
    """Reconstrói prata e ouro a partir do bronze.

    🔑 IDEMPOTENTE: rodar duas vezes dá o mesmo resultado.
    """
    partes = sorted((lago / "bronze" / f"origem={origem}").glob("data_ingestao=*"))
    selecionadas = [p for p in partes
                    if de <= date.fromisoformat(p.name.split("=")[1]) <= ate]
    if not selecionadas:
        return {"sucesso": False, "motivo": "nenhuma partição no período"}

    print(f"   reprocessando {len(selecionadas)} partição(ões):")
    quadros = []
    for parte in selecionadas:
        print(f"      {parte.name}")
        quadros.append(pd.read_parquet(parte / "dados.parquet"))

    tudo = pd.concat(quadros, ignore_index=True)
    validas, rejeitadas = validar_lote(tudo)
    prata_nova, info = construir_prata(validas)
    prata_nova.to_parquet(lago / "prata" / "vendas.parquet", index=False)

    for nome, tabela in construir_ouro(prata_nova).items():
        tabela.to_parquet(lago / "ouro" / f"{nome}.parquet", index=False)

    return {"sucesso": True, "particoes": len(selecionadas),
            "linhas_bronze": len(tudo), "linhas_prata": len(prata_nova),
            "quarentena": len(rejeitadas)}


print("\n▶ reprocessando o período inteiro\n")
r1 = reprocessar(LAGO, "erp", date(2026, 8, 13), date(2026, 8, 15))
print(f"\n   {json.dumps(r1, indent=2, ensure_ascii=False)}")

r2 = reprocessar(LAGO, "erp", date(2026, 8, 13), date(2026, 8, 15))
print(f"\n   rodou de novo → prata com {r2['linhas_prata']:,} linhas")
print(f"   ✅ idêntico: {r1['linhas_prata'] == r2['linhas_prata']}")

## 11. Linhagem e metadados

In [ ]:
def descrever_lago(lago: Path) -> pd.DataFrame:
    """Um inventário do que existe no lago.

    💭 A pergunta "de onde vem este número?" tem que ter resposta em
       um minuto. Sem inventário, cada resposta é uma investigação.
    """
    linhas = []
    for arquivo in sorted(lago.rglob("*.parquet")):
        relativo = arquivo.relative_to(lago)
        try:
            import pyarrow.parquet as pq
            meta = pq.read_metadata(arquivo)
            n_linhas, n_colunas = meta.num_rows, meta.num_columns
        except Exception:
            df = pd.read_parquet(arquivo)
            n_linhas, n_colunas = len(df), len(df.columns)
        linhas.append({
            "camada": relativo.parts[0],
            "caminho": str(relativo),
            "linhas": n_linhas,
            "colunas": n_colunas,
            "bytes": arquivo.stat().st_size,
        })
    return pd.DataFrame(linhas)


inventario = descrever_lago(LAGO)
print(inventario.to_string(index=False))
print(f"\n   total: {tamanho(inventario['bytes'].sum())}")

In [ ]:
# A ficha de uma métrica — o que resolve a discussão
FICHA = BASE / "metricas.md"
FICHA.write_text("""# Dicionário de métricas — Atlas

## receita

**Definição:** `quantidade × preco_unitario`, somado.

**Filtros:** apenas `status = 'pago'`.

**NÃO inclui:** frete, impostos, descontos aplicados após a venda.

**Fonte:** `lago/ouro/faturamento_mensal.parquet`
**Origem:** `lago/prata/vendas.parquet` ← `lago/bronze/origem=erp/`

**Fuso:** as datas são agrupadas em UTC.
⚠️ O financeiro usa horário de Brasília — esperar diferença no
   fechamento do mês (aula 10_02).

**Dono:** engenharia de dados
**Atualização:** diária, às 3h

---

## margem

**Definição:** `quantidade × (preco_unitario - custo_unitario)`, somado.

**⚠️ Cuidado:** usa o custo REGISTRADO NA VENDA, não o custo atual do
produto. Recalcular com o custo atual dá outro número — e o histórico
mudaria a cada reajuste de fornecedor (aula 10_01).

---

## pedidos

**Definição:** contagem de `pedido_id` DISTINTOS.

⚠️ Um pedido com 3 itens conta como 1 pedido e 3 itens. Confundir os
dois é a origem mais comum de divergência entre relatórios.
""", encoding="utf-8")

print(FICHA.read_text(encoding="utf-8")[:900])

> 🎯 **O dicionário de métricas é o artefato mais subestimado da engenharia de dados.**
>
> Quase toda discussão sobre "de quem é o número certo" acaba em uma destas quatro linhas:
>
> | Pergunta | Onde diverge |
> |----------|--------------|
> | O frete entra na receita? | definição |
> | Pedido pendente conta? | filtro |
> | O dia é UTC ou local? | fuso |
> | "Pedidos" são pedidos ou itens? | granularidade |
>
> 💭 **Com a ficha escrita, a conversa dura cinco minutos.** Sem ela, dura uma reunião — e volta no mês seguinte.

## 📝 Exercícios

**E1.** Explique a diferença entre ETL e ELT e por que o ELT se tornou comum.

**E2.** Monte as pastas do lago com as cinco camadas. Justifique cada uma.

**E3.** 🔑 Implemente `gravar_bronze` com manifesto e hash. Rode duas vezes e prove que não acumula.

**E4.** Escreva um contrato Pydantic para os seus dados, com normalização em `mode="before"`.

**E5.** 🔴 Implemente a quarentena. Mostre que uma linha ruim não derruba o lote nem some.

**E6.** Liste os motivos de rejeição do seu lote e proponha uma correção para os três mais frequentes.

**E7.** Implemente a deduplicação por chave natural com `keep='last'`. Explique por que ordenar antes.

**E8.** 🎯 Implemente as seis verificações de qualidade. Provoque cada falha.

**E9.** 🔑 Explique por que a verificação de volume pega o que as outras cinco não pegam.

**E10.** Distinga verificações de gravidade `erro` e `aviso`. Dê dois exemplos de cada.

**E11.** 🔴 Faça o pipeline abortar antes de publicar o ouro quando a qualidade reprova. Explique por que isso é melhor que publicar.

**E12.** Implemente a conciliação entre prata e ouro com `assert`. Quebre-a de propósito.

**E13.** 🔑 Implemente o reprocessamento a partir do bronze. Rode duas vezes e prove a idempotência.

**E14.** Escreva o inventário do seu lago com linhas, colunas e tamanho por arquivo.

**E15.** Escreva a ficha de três métricas do Atlas, incluindo o que elas NÃO incluem.

**E16.** 🔴 Simule o cenário da Aurora: uma linha ruim na 40.212ª posição. Mostre o pipeline sobrevivendo.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

In [ ]:
# E16

## 📋 Cola de referência

```python
# ═══ Camadas ═══
bronze/origem=X/data_ingestao=AAAA-MM-DD/   🔑 IMUTÁVEL + manifesto
prata/                                       limpo, tipado, deduplicado
ouro/                                        agregado por pergunta
quarentena/data=AAAA-MM-DD/                  🔴 rejeitadas COM O MOTIVO
estado/                                      marca d'água, histórico

# ═══ 🔴 Idempotência ═══
# bronze: substitui a partição, não acumula
# prata:  dedup por chave natural + sort por recência
# carga:  UPSERT com regra de desempate (aula 10_04)

# ═══ Contrato (Pydantic) ═══
class Registro(BaseModel):
    quantidade: int = Field(gt=0, le=1000)
    @field_validator("sku", mode="before")     # normaliza ANTES
    def limpar(cls, v): return v.strip().upper()

# ═══ 🔴 Quarentena ═══
try:    validas.append(Registro(**r).model_dump())
except ValidationError as e:
    rejeitadas.append({**r, "_motivo": ..., "_rejeitado_em": ...})
# 🔴 linha ruim NUNCA derruba o lote e NUNCA some

# ═══ 🎯 As 6 verificações ═══
# 1 completude · 2 unicidade · 3 não-nulos · 4 faixa
# 5 coerência entre colunas · 6 🔑 VOLUME vs histórico
# gravidade: erro = aborta · aviso = olhe isto

# ═══ Portão ═══
if falhas: return  # 🔴 NÃO publica o ouro
# dado velho e certo > dado novo e errado

# ═══ Conciliação ═══
assert abs(soma_prata - soma_ouro) < 0.01

# ═══ Reprocessar ═══
# ler bronze do período → transformação corrigida → sobrescrever
# 🔑 só é possível porque o bronze é imutável

# ═══ Linhagem ═══
# manifesto por partição · inventário do lago · ficha de cada métrica
```

## ✅ Checklist de saída

**Arquitetura**

- [ ] Sei a diferença entre ETL e ELT
- [ ] 🔑 **Meu bronze é imutável e particionado**
- [ ] Gravo manifesto com hash e contagem
- [ ] Cada camada tem um papel claro

**Idempotência**

- [ ] 🔴 **Rodar o pipeline duas vezes dá o mesmo resultado**
- [ ] Deduplico por chave natural, ordenando antes
- [ ] Sei reprocessar um período a partir do bronze

**Qualidade**

- [ ] Tenho um contrato explícito dos dados
- [ ] 🔴 **Linha ruim vai para a quarentena, com o motivo**
- [ ] Uma linha ruim não derruba o lote
- [ ] 🎯 **Implemento as seis verificações**
- [ ] 🔑 **Comparo o volume com o histórico**
- [ ] Distingo `erro` de `aviso`
- [ ] 🔴 **O pipeline não publica o ouro se a prata reprovou**
- [ ] Concilio as camadas com `assert`

**Documentação**

- [ ] Tenho inventário do lago
- [ ] 🎯 **Cada métrica tem ficha, com o que ela NÃO inclui**
- [ ] Registro o fuso usado no agrupamento

---

### ➡️ Próxima aula

**`10_06_Filas_Mensageria_Orquestracao.ipynb`** — Fazer o pipeline rodar sozinho, na ordem certa, e o que fazer quando uma etapa demora demais para caber numa requisição.